# EY Frog Challenge Feature Build

Run this notebook on Kaggle CPU with internet enabled. It queries TerraClimate, builds the engineered feature table for the union of train and test coordinates, and writes the canonical parquet artifacts.

In [ ]:
from pathlib import Path

def find_requirements_path() -> Path:
    candidates = [Path.cwd() / 'requirements-kaggle.txt', Path('/kaggle/working') / 'requirements-kaggle.txt']
    if Path('/kaggle/input').exists():
        candidates.extend(Path('/kaggle/input').glob('*/requirements-kaggle.txt'))
    for candidate in candidates:
        if candidate.exists():
            return candidate
    raise FileNotFoundError('Could not locate requirements-kaggle.txt')

REQUIREMENTS_PATH = find_requirements_path()
print(REQUIREMENTS_PATH)
!pip install -q -r "{REQUIREMENTS_PATH}"

In [ ]:
from pathlib import Path
import json
import sys

def find_project_root() -> Path:
    candidates = [Path.cwd(), Path('/kaggle/working')]
    if Path('/kaggle/input').exists():
        candidates.extend(Path('/kaggle/input').glob('*'))
    for candidate in candidates:
        if (candidate / 'feature_build.py').exists() and (candidate / 'frog_challenge').exists():
            return candidate
    raise FileNotFoundError('Could not find project root containing feature_build.py')

def find_data_root(project_root: Path) -> Path:
    candidates = [project_root, Path('/kaggle/working')]
    if Path('/kaggle/input').exists():
        candidates.extend(Path('/kaggle/input').glob('*'))
    for candidate in candidates:
        if (candidate / 'Training_Data.csv').exists() and (candidate / 'Test.csv').exists():
            return candidate
    raise FileNotFoundError('Could not find Training_Data.csv and Test.csv')

PROJECT_ROOT = find_project_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

INPUT_DIR = find_data_root(PROJECT_ROOT)
FEATURE_DIR = Path('/kaggle/working/artifacts/features')
PROJECT_ROOT, INPUT_DIR, FEATURE_DIR

In [ ]:
!python "{PROJECT_ROOT / 'feature_build.py'}" --train-path "{INPUT_DIR / 'Training_Data.csv'}" --test-path "{INPUT_DIR / 'Test.csv'}" --output-dir "{FEATURE_DIR}"

In [ ]:
manifest = json.loads((FEATURE_DIR / 'feature_manifest.json').read_text())
manifest